# Uniform Bragg Grating Reflectivity

This notebook calculates the complex reflection coefficient of a uniform Bragg grating using coupled-mode theory.

The parameters can be changed in the cell below to explore how grating strength, grating length, effective index, losses, and Bragg wavelength affect the reflection spectrum.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## Grating parameters

- `kappa`: grating coupling coefficient, in $\mu\mathrm{m}^{-1}$
- `Lg`: grating length, in $\mu\mathrm{m}$
- `neff`: effective refractive index
- `alpha_m`: material loss coefficient, in $\mu\mathrm{m}^{-1}$
- `lambda_B`: Bragg wavelength, in $\mu\mathrm{m}$
- `dlambda`: half-width of the wavelength interval, in $\mu\mathrm{m}$


In [ ]:
# Input grating parameters
kappa = 100e-4      # [um^-1]
Lg = 250            # [um]
neff = 3.25         # [-]
alpha_m = 0e-4      # [um^-1]
lambda_B = 1.55     # [um]
dlambda = 0.01      # [um]

N = 10000           # number of wavelength samples


## Coupled-mode model

The complex detuning is

$$
\delta = 2\pi n_\mathrm{eff}\left(\frac{1}{\lambda}-\frac{1}{\lambda_B}\right)-j\alpha_m,
$$

with

$$
\sigma = \sqrt{\kappa^2-\delta^2}.
$$

For a uniform grating of length $L_g$, the complex reflection coefficient is

$$
r=-j\kappa\frac{\tanh(\sigma L_g)}{\sigma+j\delta\tanh(\sigma L_g)}.
$$


In [ ]:
wavelength = np.linspace(lambda_B - dlambda, lambda_B + dlambda, N)
delta_lambda_nm = (wavelength - lambda_B) * 1000

delta = 2*np.pi*neff*(1/wavelength - 1/lambda_B) - 1j*alpha_m
sigma = np.sqrt(kappa**2 - delta**2 + 0j)

tanh_sigma_L = np.tanh(sigma * Lg)
r = -1j * kappa * tanh_sigma_L / (sigma + 1j*delta*tanh_sigma_L)

R = np.abs(r)**2


## Reflection spectrum

The first plot shows both the field reflection amplitude $|r|$ and the power reflectivity $R=|r|^2$. The second plot shows the unwrapped phase of the reflection coefficient.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(delta_lambda_nm, np.abs(r), label=r'$|r|$')
ax.plot(delta_lambda_nm, R, label=r'$R=|r|^2$')
ax.set_xlabel(r'$\lambda-\lambda_B$ [nm]')
ax.set_ylabel('Reflection')
ax.set_xlim(delta_lambda_nm[0], delta_lambda_nm[-1])
ax.set_ylim(0, 1.05)
ax.grid(True)
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(delta_lambda_nm, np.unwrap(np.angle(r))/np.pi)
ax.set_xlabel(r'$\lambda-\lambda_B$ [nm]')
ax.set_ylabel(r'$\angle r / \pi$')
ax.set_xlim(delta_lambda_nm[0], delta_lambda_nm[-1])
ax.grid(True)
plt.show()


## Normalized detuning

This reproduces the representation often used in the analytical equations, with the horizontal axis given by $\mathrm{Re}(\delta)L_g/\pi$.


In [ ]:
delta_L_over_pi = np.real(delta) * Lg / np.pi

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(delta_L_over_pi, np.abs(r))
ax.set_xlabel(r'$\mathrm{Re}(\delta)L_g/\pi$')
ax.set_ylabel(r'$|r|$')
ax.set_ylim(0, 1.05)
ax.grid(True)
ax.invert_xaxis()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(delta_L_over_pi, np.unwrap(np.angle(r))/np.pi)
ax.set_xlabel(r'$\mathrm{Re}(\delta)L_g/\pi$')
ax.set_ylabel(r'$\angle r / \pi$')
ax.grid(True)
ax.invert_xaxis()
plt.show()


## Interactive exploration

Use the sliders below to explore how the reflection spectrum changes with:

- grating coupling coefficient $\kappa$;
- grating length $L_g$;
- effective refractive index $n_\mathrm{eff}$;
- material losses $\alpha_m$;
- Bragg wavelength $\lambda_B$.

The plots are updated automatically. The first panel shows the power reflectivity $R=|r|^2$; the second shows the reflection phase.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

def plot_grating(kappa_1e4=100, Lg=250, neff=3.25, alpha_1e4=0, lambda_B=1.55, span_nm=10):
    # Slider values for kappa and alpha are expressed in units of 1e-4 um^-1
    kappa = kappa_1e4 * 1e-4
    alpha_m = alpha_1e4 * 1e-4
    dlambda = span_nm / 1000

    wavelength = np.linspace(lambda_B - dlambda, lambda_B + dlambda, 4000)
    delta_lambda_nm = (wavelength - lambda_B) * 1000

    delta = 2*np.pi*neff*(1/wavelength - 1/lambda_B) - 1j*alpha_m
    sigma = np.sqrt(kappa**2 - delta**2 + 0j)
    tanh_sigma_L = np.tanh(sigma * Lg)
    r = -1j * kappa * tanh_sigma_L / (sigma + 1j*delta*tanh_sigma_L)
    R = np.abs(r)**2

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(delta_lambda_nm, R)
    ax.set_xlabel(r'$\lambda-\lambda_B$ [nm]')
    ax.set_ylabel(r'$R=|r|^2$')
    ax.set_ylim(0, 1.05)
    ax.grid(True)
    ax.set_title(fr'$\kappa L_g={kappa*Lg:.2f}$,  $R(\lambda_B)\approx {np.tanh(kappa*Lg)**2:.3f}$ (lossless reference)')
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(delta_lambda_nm, np.unwrap(np.angle(r))/np.pi)
    ax.set_xlabel(r'$\lambda-\lambda_B$ [nm]')
    ax.set_ylabel(r'$\angle r/\pi$')
    ax.grid(True)
    plt.show()

controls = {
    'kappa_1e4': widgets.FloatSlider(value=100, min=10, max=300, step=5,
                                      description=r'$\kappa\times10^4$', continuous_update=False),
    'Lg': widgets.FloatSlider(value=250, min=50, max=1000, step=25,
                              description=r'$L_g$ [$\mu$m]', continuous_update=False),
    'neff': widgets.FloatSlider(value=3.25, min=1.4, max=4.0, step=0.05,
                                description=r'$n_{eff}$', continuous_update=False),
    'alpha_1e4': widgets.FloatSlider(value=0, min=0, max=100, step=5,
                                     description=r'$\alpha_m\times10^4$', continuous_update=False),
    'lambda_B': widgets.FloatSlider(value=1.55, min=1.2, max=1.7, step=0.01,
                                    description=r'$\lambda_B$ [$\mu$m]', continuous_update=False),
    'span_nm': widgets.FloatSlider(value=10, min=2, max=30, step=1,
                                   description='span [nm]', continuous_update=False)
}

ui = widgets.VBox(list(controls.values()))
out = widgets.interactive_output(plot_grating, controls)
display(ui, out)


### Useful quantity at exact Bragg resonance

For a lossless uniform grating, at $\lambda=\lambda_B$,

$$
|r(\lambda_B)|=\tanh(\kappa L_g),
\qquad
R(\lambda_B)=\tanh^2(\kappa L_g).
$$


In [ ]:
print(f'kappa * Lg = {kappa*Lg:.3f}')
print(f'|r(lambda_B)| (lossless formula) = {np.tanh(kappa*Lg):.6f}')
print(f'R(lambda_B) (lossless formula) = {np.tanh(kappa*Lg)**2:.6f}')
